In [ ]:
from collections import defaultdict
import json
import os

from torchvision.transforms import v2
from skimage import filters
import matplotlib.pyplot as plt
import numpy
import pandas
import torch
import tqdm


MODEL_NB = 618
VIRTUES_MODEL = 'virtues_our_norm'

REC_PATH_VIRTUES = f'path/to/dataset/recons/{VIRTUES_MODEL}/'
REC_PATH_IMMU = f'path/to/dataset/results/with_reconstructs/recons/immuvis_{MODEL_NB}/'
GT_PATH_IMMU = f'path/to/dataset/results/with_reconstructs/recons/ground_truth/'

IMMU_OFFSET = 3.4
IMMU_LOG1P_OFFSET = 5.4

PLOT = True

In [ ]:
print(f"Configuration set for Models: {MODEL_NB} | {VIRTUES_MODEL}")

def our_center_crop(img_, crop_size=(128, 128)):
    center_x = img_.shape[0] // 2
    center_y = img_.shape[1] // 2
    crop_span_x = crop_size[0] // 2
    crop_span_y = crop_size[1] // 2
    return img_[
        center_x - crop_span_x : center_x + crop_span_x,
        center_y - crop_span_y : center_y + crop_span_y,
    ]

def vector_mse(x, y):
    return ((x.flatten() - y.flatten()) ** 2).mean()

virtues_paths = [
    path for path in os.listdir(REC_PATH_VIRTUES)
    if path.endswith('recon.npy')
]

virtues_to_npy = dict()
indices = []
markers = []

print("\n--- Loading Datasets ---")
for path in tqdm.tqdm(virtues_paths, desc="Loading VIRTUES Predictions"):
    full_path = os.path.join(REC_PATH_VIRTUES, path)
    img_index, marker = path.split('_')[:2]
    indices.append(img_index)
    markers.append(marker)
    virtues_to_npy[(img_index, marker)] = numpy.load(full_path)

def load_immu_data(folder_path):
    data_dict = dict()
    for path in os.listdir(folder_path):
        if not path.endswith('npz'):
            continue
        immu_path = f'{folder_path}/{path}'
        result = numpy.load(immu_path)
        if not json.loads(result['metadata'].item())['dataset_name'] == 'hn':
            continue
        img_index = json.loads(result['metadata'].item())['image_path'].split('/')[-1].split('.')[0]
        data_dict[img_index] = result
    return data_dict

print("Loading IMMUVIS Predictions & Ground Truth...")
immu_to_npy = load_immu_data(REC_PATH_IMMU)
immu_to_npy_gt = load_immu_data(GT_PATH_IMMU)

markers_list = [str(el) for el in list(immu_to_npy.values())[0]['marker_names']]

print("Loading Marker Normalization Statistics...")
marker_stats = pandas.read_csv('path/to/marker_ds_stats.csv')
marker_to_mean, marker_to_std = {}, {}

for _, row in marker_stats.iterrows():
    if row['dataset'] == 'hn':
        marker_to_mean[row['marker']] = row['mean']
        marker_to_std[row['marker']] = row['std']

print("Data loading complete.")

In [ ]:
print("\n--- Evaluating Image-Level Performance ---")
print(f"Comparing Us IMMUVIS vs. {VIRTUES_MODEL}..")

immuvis_mses, immuvis_pearsons = [], []
virtues_mses, virtues_pearsons = [], []

imgs_, markers_ = [], []

gts = defaultdict(list)
gts_squared = defaultdict(list)
immuvis_preds_total = defaultdict(list)
virtues_preds_total = defaultdict(list)
immuvis_pred_gt_total = defaultdict(list)
virtues_pred_gt_total = defaultdict(list)
immuvis_preds_squared_total = defaultdict(list)
virtues_preds_squared_total = defaultdict(list)

for tup_ in tqdm.tqdm(virtues_to_npy, desc="Processing Images"):
    img, marker = tup_
    
    if marker == 'Carbonic':
        marker = 'Carbonic Anhydrase'
    if marker == 'PARP':
        marker = 'cl.PARP'
    if marker in ['H3', 'Histone']:
        marker = 'Histone H3'

    imgs_.append(img)
    markers_.append(marker)
        
    marker_immu_index = list(immu_to_npy[img]['marker_names']).index(marker)
    virtues_preds_img = virtues_to_npy[tup_]
    
    model_name_lower = VIRTUES_MODEL.lower()
    
    if 'eva' in model_name_lower:
        # EVA reconstructions were generated using ImmuVis preprocessing
        virtues_preds = virtues_preds_img.flatten() * IMMU_OFFSET
        immu_preds = immu_to_npy[img]['recon'][marker_immu_index].flatten() * IMMU_OFFSET
        marker_img = immu_to_npy[img]['target'][marker_immu_index] * IMMU_OFFSET

    elif 'virtues' in model_name_lower:
        if 'zs' in model_name_lower:
            virtues_preds = (numpy.log1p(numpy.sinh(virtues_preds_img.flatten()) * 5.) - marker_to_mean[marker]) / marker_to_std[marker]
            
            if MODEL_NB not in list(range(654, 660)):
                immu_preds = immu_to_npy[img]['recon'][marker_immu_index].flatten()
                marker_img = immu_to_npy[img]['target'][marker_immu_index]
            else:
                immu_preds = immu_to_npy[img]['recon'][marker_immu_index].flatten() * IMMU_LOG1P_OFFSET
                immu_preds = (immu_preds - marker_to_mean[marker]) / marker_to_std[marker]
                marker_img = immu_to_npy_gt[img]['target'][marker_immu_index]
        else:
            # virtues_our_norm
            virtues_preds = virtues_preds_img.flatten()
            immu_preds = immu_to_npy[img]['recon'][marker_immu_index].flatten() * IMMU_OFFSET
            marker_img = immu_to_npy[img]['target'][marker_immu_index] * IMMU_OFFSET
            
    else:
        # immuvis
        virtues_preds = virtues_preds_img.flatten()
        immu_preds = immu_to_npy[img]['recon'][marker_immu_index].flatten() * IMMU_OFFSET
        marker_img = immu_to_npy[img]['target'][marker_immu_index] * IMMU_OFFSET

    if PLOT:
        plt.subplot(1, 3, 1)
        plt.imshow(immu_preds.reshape((128, 128)))
        plt.subplot(1, 3, 2)
        plt.imshow(marker_img)
        plt.subplot(1, 3, 3)
        plt.imshow(virtues_preds.reshape((128, 128)))
        plt.show()

    crop = our_center_crop(marker_img)

    virtues_mse = vector_mse(virtues_preds, crop)
    virtues_mses.append(virtues_mse)

    gts[marker].append(crop.mean())
    gts_squared[marker].append((crop ** 2).mean())
    
    virtues_preds_total[marker].append(virtues_preds.mean())
    virtues_preds_squared_total[marker].append((virtues_preds ** 2).mean())
    virtues_pred_gt_total[marker].append((virtues_preds * crop.flatten()).mean())

    virtues_corr_coef = numpy.corrcoef(virtues_preds, crop.flatten())[0, 1]
    virtues_pearsons.append(virtues_corr_coef)

    immuvis_preds_total[marker].append(immu_preds.mean())
    immuvis_preds_squared_total[marker].append((immu_preds ** 2).mean())
    immuvis_pred_gt_total[marker].append((immu_preds * crop.flatten()).mean())

    immu_mse = vector_mse(crop, immu_preds)
    immuvis_mses.append(immu_mse)

    immuvis_pearson = numpy.corrcoef(immu_preds.flatten(), crop.flatten())[0, 1]
    immuvis_pearsons.append(immuvis_pearson)

print("Metrics collection complete.")

In [ ]:
print("Computing Global Pearson Correlation per Marker")
immu_pearson_total = {}
virtues_pearson_total = {}
pearson_difference = {}

for marker in immuvis_pred_gt_total:
    # IMMUVIS
    mean_immu = numpy.mean(immuvis_preds_total[marker])
    mean_gt = numpy.mean(gts[marker])
    mean_cross_immu = numpy.mean(immuvis_pred_gt_total[marker])

    sq_gt = (numpy.mean(gts_squared[marker]) - mean_gt ** 2) ** 0.5
    sq_immu = (numpy.mean(immuvis_preds_squared_total[marker]) - mean_immu ** 2) ** 0.5

    immu_pearson_total[marker] = (mean_cross_immu - mean_gt * mean_immu) / (sq_gt * sq_immu)

    # VIRTUES
    mean_virtues = numpy.mean(virtues_preds_total[marker])
    mean_cross_virtues = numpy.mean(virtues_pred_gt_total[marker])
    
    sq_virtues = (numpy.mean(virtues_preds_squared_total[marker]) - mean_virtues ** 2) ** 0.5

    virtues_pearson_total[marker] = (mean_cross_virtues - mean_gt * mean_virtues) / (sq_gt * sq_virtues)

    pearson_difference[marker] = immu_pearson_total[marker] - virtues_pearson_total[marker]

pearson_df = pandas.DataFrame({
    'marker': list(immu_pearson_total.keys()),
    'immuvis': list(immu_pearson_total.values()),
    'virtues': [virtues_pearson_total[key] for key in immu_pearson_total],
})
pearson_df.to_csv(f'VIRTUAL_STAINING_PEARSON_{MODEL_NB}_{VIRTUES_MODEL}.csv', index=False)

results_df = pandas.DataFrame({
    'img': imgs_,
    'marker': markers_,
    'immuvis_pearson': immuvis_pearsons,
    'immuvis_mse': immuvis_mses,
    'virtues_pearson': virtues_pearsons,
    'virtues_mse': virtues_mses,
})

results_df.to_csv(f'VIRTUAL_STAINING_{MODEL_NB}_{VIRTUES_MODEL}.csv', index=False)

results_df_all = results_df.copy()
results_df_all['marker'] = 'All'
results_df_full = pandas.concat([results_df, results_df_all], ignore_index=True)

check_mean = results_df_full.groupby('marker')[['immuvis_pearson', 'immuvis_mse', 'virtues_pearson', 'virtues_mse']].mean()
check_std = results_df_full.groupby('marker')[['immuvis_pearson', 'immuvis_mse', 'virtues_pearson', 'virtues_mse']].std()

print("\n" + "="*50)
print(f"RESULTS SUMMARY: IMMUVIS_{MODEL_NB} | {VIRTUES_MODEL}")
print("="*50)

print(f"Mean Global Pearson Difference (IMMUVIS - VIRTUES): {numpy.mean(list(pearson_difference.values())):.4f}")

print("\n--- Image-Level Overall ('All') Performance ---")
print(f"IMMUVIS MSE:     {check_mean['immuvis_mse']['All']:.4f} ± {check_std['immuvis_mse']['All']:.4f}")
print(f"VIRTUES MSE:      {check_mean['virtues_mse']['All']:.4f} ± {check_std['virtues_mse']['All']:.4f}")

print(f"\nIMMUVIS Pearson: {check_mean['immuvis_pearson']['All']:.4f} ± {check_std['immuvis_pearson']['All']:.4f}")
print(f"VIRTUES Pearson:  {check_mean['virtues_pearson']['All']:.4f} ± {check_std['virtues_pearson']['All']:.4f}")
print("="*50)